# 0. Import des données 

In [12]:
import duckdb
import random
import numpy as np
import pandas as pd

# 1. Repertoire de travail 

In [ ]:
full_data = pd.read_parquet("/home/user_daren_mabanza/data_science/projets/V2_projet_fraud_detection/01_data/01_raw_data/FullData.parquet")

In [115]:
full_data_copy = full_data.copy()

# 2. Nettoyage des données 

In [64]:
full_data_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 23 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   column00               int64         
 1   trans_date_trans_time  datetime64[ns]
 2   cc_num                 int64         
 3   merchant               object        
 4   category               object        
 5   amt                    float64       
 6   first                  object        
 7   last                   object        
 8   gender                 object        
 9   street                 object        
 10  city                   object        
 11  state                  object        
 12  zip                    int64         
 13  lat                    float64       
 14  long                   float64       
 15  city_pop               int64         
 16  job                    object        
 17  dob                    object        
 18  trans_num             

In [67]:
full_data_copy.sample(7)

,column00,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1174779,792173,2019-12-02 10:32:08,2224100459250597,fraud_Kling Inc,gas_transport,31.74,Sheri,Thomas,F,019 Kimberly Light Apt. 039,...,30.7906,-81.5607,1293,Naval architect,1963-09-11,fdd4661102b3231c529798857cfb7404,1354444328,31.542167,-81.253509,0
120307,1280302,2020-06-15 11:39:55,3546897637165774,fraud_McGlynn-Heathcote,misc_net,77.48,Kayla,Obrien,F,7921 Robert Port Suite 343,...,31.6591,-96.8094,263,Barrister,1956-05-30,7e3454f2ff9c09918e9d33374864a500,1371296395,31.653464,-97.093193,0
1175528,987879,2020-02-07 06:06:40,4102003771126577611,fraud_Koepp-Parker,grocery_pos,56.80,William,Fitzgerald,M,715 Courtney Pike Suite 932,...,32.9276,-97.2489,95035,Probation officer,1987-06-13,edaf1dd695c9fcc0e9e16fcd1d832e11,1360217200,32.787598,-97.022513,0
914364,390426,2020-11-21 13:41:39,4048508450704760,"fraud_Lang, Towne and Schuppe",kids_pets,53.91,Cheryl,Melton,F,5053 Bell Crescent Apt. 255,...,41.3660,-98.0054,1749,"Surveyor, mining",1973-05-07,ff1cc298b6c9e8d09168fac73958a270,1385041299,41.728393,-97.472027,0
161218,905896,2019-12-28 10:23:34,3577794103155425,fraud_Bauch-Raynor,grocery_pos,94.79,Mary,Humphrey,F,9853 Flynn Hollow Apt. 602,...,47.7068,-97.2092,733,Visual merchandiser,1994-11-12,5f75b53636c57f8abbfc996f2417758f,1356690214,48.464492,-96.744579,0
1501713,4460,2019-01-03 19:12:48,376262134119629,"fraud_Cronin, Kshlerin and Weber",health_fitness,50.55,Christopher,Carr,M,62931 Ralph Court,...,27.9611,-81.9398,237282,"Radiographer, therapeutic",1961-09-28,6b006ae5719dbc9ff9ca0b4f60d4874b,1325617968,27.917949,-82.333428,0
1143546,198155,2019-04-12 13:17:27,4961003488432306,"fraud_Hills, Hegmann and Schaefer",health_fitness,15.87,Stacy,Walsh,F,324 Donna Ridge Suite 651,...,40.6284,-76.1950,1606,Ecologist,1966-05-10,2c67f0e48d2463834dd7a76b8a50d3ce,1334236647,39.978344,-75.475023,0


In [123]:
full_data_copy.isna().mean()

id                       0.0
trans_date_trans_time    0.0
merchant                 0.0
category                 0.0
amt                      0.0
city                     0.0
state                    0.0
lat                      0.0
long                     0.0
city_pop                 0.0
job                      0.0
dob                      0.0
merch_lat                0.0
merch_long               0.0
is_fraud                 0.0
dtype: float64

In [116]:
# column00
full_data_copy["column00"] = 1
full_data_copy["column00"] = full_data_copy["column00"].cumsum()
full_data_copy["column00"] = full_data_copy["column00"].astype("uint32")
full_data_copy = full_data_copy.rename(columns={"column00":"id"})

# trans_date_trans_time : OK

# cc_num 
full_data_copy = full_data_copy.drop(["cc_num"], axis=1)

# merchant
full_data_copy["merchant"] = full_data_copy["merchant"].str.slice_replace(0,6,"")
full_data_copy["merchant"] = full_data_copy["merchant"].astype("category")

# category
full_data_copy["category"] = full_data_copy["category"].astype("category")

# amt
full_data_copy["amt"] = full_data_copy["amt"].astype("float16")

# first 
full_data_copy = full_data_copy.drop(["first"], axis=1)

# last
full_data_copy = full_data_copy.drop(["last"], axis=1)

# gender 
full_data_copy = full_data_copy.drop(["gender"], axis=1)

# street
full_data_copy = full_data_copy.drop(["street"], axis=1)

# city
full_data_copy["city"] = full_data_copy["city"].astype("category")

# state
full_data_copy["state"] = full_data_copy["state"].astype("category")

# zip
full_data_copy = full_data_copy.drop(["zip"], axis=1)

# lat
full_data_copy["lat"] = full_data_copy["lat"].astype("float32")

# long
full_data_copy["long"] = full_data_copy["long"].astype("float32")

# city_pop
full_data_copy["city_pop"] = full_data_copy["city_pop"].astype("uint32")

# job
full_data_copy["job"] = full_data_copy["job"].astype("category")

# dob
full_data_copy["dob"] = pd.to_datetime(full_data_copy["dob"])

# trans_num
full_data_copy = full_data_copy.drop(["trans_num"], axis=1)

# unix_time
full_data_copy = full_data_copy.drop(["unix_time"], axis=1)

# merch_lat
full_data_copy["merch_lat"] = full_data_copy["merch_lat"].astype("float32")

# merch_long
full_data_copy["merch_long"] = full_data_copy["merch_long"].astype("float32")

# is_fraud
full_data_copy["is_fraud"] = full_data_copy["is_fraud"].astype("uint8")

In [117]:
full_data_copy.sample(7)

,id,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,merch_lat,merch_long,is_fraud
382135,382136,2019-03-12 15:57:10,Gislason Group,travel,1.330078,Eugene,OR,44.038502,-123.061401,191096,"Scientist, physiological",1964-04-06,44.756721,-122.159760,0
1409905,1409906,2020-06-16 16:20:32,Padberg-Sauer,home,74.937500,Oconto Falls,WI,44.875500,-88.155502,5548,Mining engineer,1975-09-11,44.670223,-89.008842,0
1026195,1026196,2020-08-25 21:56:19,Lynch-Wisozk,home,17.390625,Grenada,CA,41.612499,-122.525803,589,Systems analyst,1945-12-21,42.086323,-123.163467,0
222716,222717,2019-08-17 23:49:22,Howe Ltd,misc_pos,5.230469,Providence,RI,41.838299,-71.437698,203571,"Nurse, children's",1995-11-29,40.996967,-71.590141,0
1531675,1531676,2019-05-30 14:25:47,"Kutch, Steuber and Gerhold",food_dining,45.031250,Diamond,OH,41.093498,-81.042503,2644,"Engineer, biomedical",1983-08-25,40.873856,-80.429459,0
507711,507712,2020-05-21 02:28:35,Kuhn LLC,shopping_pos,9.570312,Tampa,FL,28.059999,-82.407898,717255,Waste management officer,1966-09-16,27.132917,-83.385704,0
973517,973518,2020-10-17 08:39:12,"Romaguera, Cruickshank and Greenholt",shopping_net,81.812500,Egan,LA,30.250999,-92.500198,1261,Broadcast presenter,1972-07-01,29.497726,-91.931099,0


In [118]:
full_data_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 15 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id                     uint32        
 1   trans_date_trans_time  datetime64[ns]
 2   merchant               category      
 3   category               category      
 4   amt                    float16       
 5   city                   category      
 6   state                  category      
 7   lat                    float32       
 8   long                   float32       
 9   city_pop               uint32        
 10  job                    category      
 11  dob                    datetime64[ns]
 12  merch_lat              float32       
 13  merch_long             float32       
 14  is_fraud               uint8         
dtypes: category(5), datetime64[ns](2), float16(1), float32(4), uint32(2), uint8(1)
memory usage: 90.2 MB
